# Apps Migration: Review and Approve

This notebook allows you to review the discovered apps and select which ones to migrate.

**Input:** `apps_inventory.csv` from inventory generation
**Output:** `inventory_approved.csv` for export step

## Setup

In [ ]:
from databricks.sdk import WorkspaceClient
import pandas as pd

w = WorkspaceClient()

## Configuration

In [ ]:
catalog = dbutils.widgets.get("catalog") if "catalog" in dbutils.widgets.getAll() else "YOUR_SOURCE_CATALOG"
schema = dbutils.widgets.get("schema") if "schema" in dbutils.widgets.getAll() else "apps_demo"
volume = dbutils.widgets.get("volume") if "volume" in dbutils.widgets.getAll() else "apps_migration"

volume_path = f"/Volumes/{catalog}/{schema}/{volume}"
inventory_path = f"{volume_path}/apps_inventory/apps_inventory.csv"
approved_path = f"{volume_path}/apps_inventory_approved/inventory_approved.csv"

print(f"Reading inventory from: {inventory_path}")
print(f"Will write approved list to: {approved_path}")

## Read inventory

In [ ]:
import csv
from io import StringIO

response = w.files.download(inventory_path)
content = response.contents.read().decode('utf-8')

df = pd.read_csv(StringIO(content))
print(f"Loaded {len(df)} apps from inventory")

## Display inventory for review

In [ ]:
print("=" * 60)
print("APPS INVENTORY FOR REVIEW")
print("=" * 60)
print("")

display(df[["name", "description", "status", "creator", "env_var_count"]])

## Filter apps to migrate

Modify the `apps_to_migrate` list below to select which apps to include in the migration.

Options:
- Set to `None` or `[]` to migrate ALL apps
- Set to a list of app names to migrate only those apps
- Use `exclude_apps` to exclude specific apps

In [ ]:
apps_to_migrate = None

exclude_apps = []

if apps_to_migrate:
    approved_df = df[df['name'].isin(apps_to_migrate)].copy()
else:
    approved_df = df.copy()

if exclude_apps:
    approved_df = approved_df[~approved_df['name'].isin(exclude_apps)]

print(f"Apps selected for migration: {len(approved_df)}")
for name in approved_df['name'].tolist():
    print(f"  - {name}")

## Confirmation step

Type `CONFIRM` in the widget below to proceed with saving the approved list.

In [ ]:
dbutils.widgets.text("confirmation", "", "Type CONFIRM to proceed")
confirmation = dbutils.widgets.get("confirmation")

if confirmation != "CONFIRM":
    print("⚠️  Please type CONFIRM in the widget above to save the approved list.")
    print("   This step requires explicit confirmation before proceeding.")
else:
    print("✅ Confirmation received. Proceeding to save approved list.")

## Save approved inventory

In [ ]:
from io import BytesIO

if confirmation == "CONFIRM":
    csv_content = approved_df.to_csv(index=False)
    w.files.upload(approved_path, BytesIO(csv_content.encode('utf-8')), overwrite=True)
    
    print("=" * 60)
    print("APPROVED LIST SAVED")
    print("=" * 60)
    print(f"Apps approved: {len(approved_df)}")
    print(f"File: {approved_path}")
    print("")
    print("Next step: Run the export job (Src_03_Export_Apps)")
    print("=" * 60)
else:
    print("❌ Approved list NOT saved. Please confirm first.")